In [1]:
import requests
import pandas as pd 
from bs4 import BeautifulSoup as BS

In [2]:
url = "https://books.toscrape.com/"
headers = {"User-Agent": ("Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)" "AppleWebKit/537.36 (KHTML, like Gecko)" "Chrome/127.0.0.0 Safari/537.36")}

In [3]:
books_data = []

In [4]:
for page_no in range (1, 6):
    if page_no == 1: url_00 = url
    else: url_00 = f"{url}catalogue/page-{page_no}.html"
    print("Scrapping :", url_00)
    reponse = requests.get(url_00, headers=headers)
    print("Status    :",reponse.status_code)

    if (reponse.status_code != 200): 
        print("book not found")
        continue

    soup = BS(reponse.content, "html.parser")
    books = soup.find_all("article", class_="product_pod")
    print("Book found:", len(books))

Scrapping : https://books.toscrape.com/
Status    : 200
Book found: 20
Scrapping : https://books.toscrape.com/catalogue/page-2.html
Status    : 200
Book found: 20
Scrapping : https://books.toscrape.com/catalogue/page-3.html
Status    : 200
Book found: 20
Scrapping : https://books.toscrape.com/catalogue/page-4.html
Status    : 200
Book found: 20
Scrapping : https://books.toscrape.com/catalogue/page-5.html
Status    : 200
Book found: 20


In [7]:
for book in books:
    title = book.h3.a["title"]
    price = book.find("p",class_="price_color").text.strip()
    rating = book.find("p",class_="star-rating")["class"][1]
    availability = book.find("p",class_="instock availability").text.strip()
    relative_link = book.h3.a["href"]
    full_link = url + "catalogue/" + relative_link.replace("../../../","")
    books_data.append({"title": title,"price": price,"rating":
    rating,"availability": availability,"link": full_link})

In [12]:
df = pd.DataFrame(books_data)
df.head()


,title,price,rating,availability,link
0,"Princess Jellyfish 2-in-1 Omnibus, Vol. 01 (Pr...",£13.61,Five,In stock,https://books.toscrape.com/catalogue/princess-...
1,Princess Between Worlds (Wide-Awake Princess #5),£13.34,Five,In stock,https://books.toscrape.com/catalogue/princess-...
2,"Pop Gun War, Volume 1: Gift",£18.97,One,In stock,https://books.toscrape.com/catalogue/pop-gun-w...
3,"Political Suicide: Missteps, Peccadilloes, Bad...",£36.28,Two,In stock,https://books.toscrape.com/catalogue/political...
4,Patience,£10.16,Three,In stock,https://books.toscrape.com/catalogue/patience_...


In [13]:
print("\nShape")
print(df.shape)
print("\nColumns")
print(df.columns)
print("\nData types")
print(df.dtypes)


Shape
(20, 5)

Columns
Index(['title', 'price', 'rating', 'availability', 'link'], dtype='object')

Data types
title           object
price           object
rating          object
availability    object
link            object
dtype: object


In [14]:
df["price"] = (df["price"].str.replace("£", "", regex=False).astype(float))
rating_map = {"One": 1,"Two": 2,"Three": 3,"Four": 4,"Five": 5}
df["rating"] = df["rating"].map(rating_map)

In [15]:
df.head()

,title,price,rating,availability,link
0,"Princess Jellyfish 2-in-1 Omnibus, Vol. 01 (Pr...",13.61,5,In stock,https://books.toscrape.com/catalogue/princess-...
1,Princess Between Worlds (Wide-Awake Princess #5),13.34,5,In stock,https://books.toscrape.com/catalogue/princess-...
2,"Pop Gun War, Volume 1: Gift",18.97,1,In stock,https://books.toscrape.com/catalogue/pop-gun-w...
3,"Political Suicide: Missteps, Peccadilloes, Bad...",36.28,2,In stock,https://books.toscrape.com/catalogue/political...
4,Patience,10.16,3,In stock,https://books.toscrape.com/catalogue/patience_...


In [16]:
df.dtypes

title            object
price           float64
rating            int64
availability     object
link             object
dtype: object

In [17]:
df.to_csv("books_dataset.csv",index=False,encoding="utf-8-sig")

In [18]:
print("\nAverage price:")
print(df["price"].mean())
print("\nAverage rating:")
print(df["rating"].mean())
print("\nMost expensive book:")
print(df.loc[df["price"].idxmax(),"title"])
print("\nHighest rated books:")
print(df[df["rating"] == 5][["title", "price", "rating"]].head())


Average price:
29.97

Average rating:
2.7

Most expensive book:
Masks and Shadows

Highest rated books:
                                                title  price  rating
0   Princess Jellyfish 2-in-1 Omnibus, Vol. 01 (Pr...  13.61       5
1    Princess Between Worlds (Wide-Awake Princess #5)  13.34       5
18                                               Join  35.67       5
